In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import RobustScaler

# 재현성을 위한 시드 고정
tf.random.set_seed(42)
np.random.seed(42)

# =====================================================================
# 1. '순백의 정상 데이터' 추출 및 스케일링 함수 (PyTorch 버전과 동일)
# =====================================================================
def prepare_normal_data(df, train_end_date, features):
    # 1. 기간 분리 (학습용: 초기 건강한 상태 / 테스트용: 이후 전체 기간)
    train_df = df[df.index <= train_end_date].copy()
    test_df = df.copy() 
    
    X_train = train_df[features].copy()
    X_test = test_df[features].copy()
    
    # 2. Train 데이터의 스파이크(이상치) 제거 (클리핑)
    for col in features:
        Q1 = X_train[col].quantile(0.25)
        Q3 = X_train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        X_train[col] = X_train[col].clip(lower=lower_bound, upper=upper_bound)

    # 3. 스케일링 (RobustScaler)
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, scaler, test_df.index

# =====================================================================
# 2 & 3. TF/Keras 모델 정의 및 학습 함수
# =====================================================================
def train_tf_autoencoder(X_train, epochs=50, batch_size=64, validation_split=0.1):
    input_dim = X_train.shape[1]
    
    # 모델 아키텍처 정의
    model = models.Sequential([
        # 인코더 부분
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(8,  activation='relu'),
        
        # 디코더 부분
        layers.Dense(16, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(input_dim, activation='linear') # 원본 복원이므로 마지막은 linear
    ])
    
    # 옵티마이저 및 손실 함수 설정
    model.compile(optimizer='adam', loss='mse')
    
    # 💡 Keras의 꽃: Early Stopping (오버피팅 방지)
    # 검증 데이터(val_loss)의 오차가 5번 연속 개선되지 않으면 학습을 조기 종료합니다.
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True
    )
    
    print("--- TensorFlow AutoEncoder 학습 시작 ---")
    # Keras는 fit() 함수 하나로 학습 루프가 끝납니다. Input과 Target 모두 X_train을 줍니다.
    history = model.fit(
        X_train, X_train, 
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split, # 데이터의 10%를 검증용으로 자동 할당
        callbacks=[early_stopping],
        verbose=1
    )
    
    return model, history

# =====================================================================
# 4. 이상 탐지 및 임계치(Threshold) 설정 함수
# =====================================================================
def detect_anomalies_tf(model, X_train_scaled, X_test_scaled, test_indices, threshold_percentile=99):
    """
    Keras 모델의 predict()를 활용하여 재구성 오차(MSE)를 계산합니다.
    """
    print("\n--- 이상 탐지 및 추론 시작 ---")
    
    # 1. Train 데이터로 완벽한 정상 상태의 '오차 한계선' 설정
    train_pred = model.predict(X_train_scaled, verbose=0)
    # Numpy를 활용한 빠르고 간단한 MSE 계산 (행별 평균 제곱 오차)
    train_mse = np.mean(np.square(X_train_scaled - train_pred), axis=1)
    
    threshold = np.percentile(train_mse, threshold_percentile)
    print(f"✅ 설정된 이상 탐지 임계치(Threshold): {threshold:.4f}")

    # 2. Test 데이터(실제 전체 가동 데이터) 이상 탐지 수행
    test_pred = model.predict(X_test_scaled, verbose=0)
    test_mse = np.mean(np.square(X_test_scaled - test_pred), axis=1)
    
    # 3. 결과 정리
    result_df = pd.DataFrame(index=test_indices)
    result_df['reconstruction_error'] = test_mse
    result_df['threshold'] = threshold
    # 오차가 임계치를 넘으면 1(비정상), 아니면 0(정상)
    result_df['is_anomaly'] = (test_mse > threshold).astype(int)
    
    return result_df